In [1]:
# Notebook 03: Modeling AI Prediksi Harga Rumah

# Library dasar
import pandas as pd
import numpy as np

# Library untuk split data & modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Library untuk evaluasi
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Biar output rapi
pd.set_option('display.max_columns', None)

print("Library modeling siap!")

Library modeling siap!


In [2]:
# Baca dataset bersih hasil preprocessing
df = pd.read_csv("../data/indonesia_housing_clean.csv")

# Cek ukuran & kolom
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {df.shape[1]}")
print(f"\nKolom-kolomnya:")
for i, col in enumerate(df.columns, 1):
    print(f" {i}. {col} ({df[col].dtype})")

Jumlah baris: 87143
Jumlah kolom: 22

Kolom-kolomnya:
 1. housing_name (str)
 2. property_type (str)
 3. subsidy_status (str)
 4. province (str)
 5. city_regency (str)
 6. district (str)
 7. village (str)
 8. latitude (float64)
 9. longitude (float64)
 10. price_idr (int64)
 11. land_area_m2 (float64)
 12. building_area_m2 (float64)
 13. bedrooms (int64)
 14. bathrooms (int64)
 15. floors (int64)
 16. roof_material (str)
 17. wall_material (str)
 18. floor_material (str)
 19. foundation_material (str)
 20. developer_name (str)
 21. developer_association (str)
 22. total_units (int64)


In [3]:
# Cek missing values di kolom yang akan dipakai
print("=== Missing values sebelum drop ===")
kolom_penting = ['land_area_m2', 'building_area_m2', 'bedrooms', 'bathrooms',
                 'floors', 'total_units', 'latitude', 'longitude',
                 'subsidy_status', 'province', 'city_regency']
missing = df[kolom_penting].isnull().sum()
print(missing[missing > 0]) if (missing > 0).any() else "Tidak ada missing values"

# Buang baris dengan NaN
df = df.dropna(subset=kolom_penting)

print(f"\nJumlah baris setelah drop: {len(df)}")

=== Missing values sebelum drop ===
latitude     75
longitude    75
dtype: int64

Jumlah baris setelah drop: 87068


In [4]:
# Kolom target (yang mau diprediksi)
TARGET = 'price_idr'

# Kolom fitur numerik (angka)
fitur_numerik = [
    'land_area_m2',
    'building_area_m2',
    'bedrooms',
    'bathrooms',
    'floors',
    'total_units',
    'latitude',
    'longitude'
]

# Kolom fitur kategorikal (teks)
fitur_kategorikal = [
    'subsidy_status',
    'province',
    'city_regency',
]

# Gabungkan semua fitur
semua_fitur = fitur_numerik + fitur_kategorikal

print(f"Target: {TARGET}")
print(f"Jumlah fitur numerik: {len(fitur_numerik)}")
print(f"Jumlah fitur kategorikal: {len(fitur_kategorikal)}")
print(f"Total fitur: {len(semua_fitur)}")

Target: price_idr
Jumlah fitur numerik: 8
Jumlah fitur kategorikal: 3
Total fitur: 11


In [5]:
# Pisahkan X (fitur) dan y (target)
X = df[semua_fitur]
y = df[TARGET]

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"\nTotal data: {len(X)}")
print(f"Training:   {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing:    {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

Training set: (69654, 11)
Testing set: (17414, 11)

Total data: 87068
Training:   69654 (80.0%)
Testing:    17414 (20.0%)


In [6]:
# Cek jumlah nilai unik di tiap kolom kategorikal
print("=== Jumlah nilai unik di kolom kategorikal ===\n")
for col in fitur_kategorikal:
    n_unique = df[col].nunique()
    print(f"{col}: {n_unique} nilai unik")

    # Tampilan 5 teratas kalau jumlahnya banyak
    if n_unique > 5:
        print(f"    Contoh: {df[col].value_counts().head(3).to_dict()}")
    print()

=== Jumlah nilai unik di kolom kategorikal ===

subsidy_status: 2 nilai unik

province: 34 nilai unik
    Contoh: {'JAWA BARAT': 23597, 'JAWA TIMUR': 12050, 'BANTEN': 10056}

city_regency: 451 nilai unik
    Contoh: {'KAB BOGOR': 5383, 'KAB BEKASI': 5019, 'KAB TANGERANG': 4387}



In [7]:
# Cek nilai unik property_type
print("Nilai unik property_type:")
print(df['property_type'].unique())
print()
print("Jumlah kemunculan:")
print(df['property_type'].value_counts())

Nilai unik property_type:
<ArrowStringArray>
['Rumah Tapak']
Length: 1, dtype: str

Jumlah kemunculan:
property_type
Rumah Tapak    87068
Name: count, dtype: int64


In [8]:
# Normalisasi kolom teks material: lowercase + strip spasi
kolom_perlu_normalisasi = [
    'roof_material',
    'wall_material',
    'floor_material',
    'foundation_material'
]

print("=== SEBELUM normalisasi ===")
for col in kolom_perlu_normalisasi:
    print(f"{col}: {df[col].nunique()} nilai unik")

# Proses normalisasi
for col in kolom_perlu_normalisasi:
    df[col] = df[col].str.lower().str.strip()

print("\n=== SESUDAH normalisasi ===")
for col in kolom_perlu_normalisasi:
    print(f"{col}: {df[col].nunique()} nilai unik")

=== SEBELUM normalisasi ===
roof_material: 19383 nilai unik
wall_material: 21864 nilai unik
floor_material: 11122 nilai unik
foundation_material: 12735 nilai unik

=== SESUDAH normalisasi ===
roof_material: 13941 nilai unik
wall_material: 16364 nilai unik
floor_material: 7832 nilai unik
foundation_material: 9004 nilai unik


In [9]:
# Buat preprocessor ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        # Kolom kategorikal -> OneHotEncoder
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), fitur_kategorikal),
        # Kolom numerik -> dibiarkan apa adanya
        ('num', 'passthrough', fitur_numerik)
    ]
)

print("Preprocessor siap!")

Preprocessor siap!


In [10]:
# Buat pipeline: prepocessor -> model Linear Regression
model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Training model
print("Training model Linear Regerssion...")
model_lr.fit(X_train, y_train)
print("Training selesai!")

Training model Linear Regerssion...
Training selesai!


In [11]:
# Prediksi harga rumah di data testing
y_pred = model_lr.predict(X_test)

# Hitung metrik evaluasi
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Hasil Evaluasi Model Linear Regression ===")
print(f"MAE:    Rp {mae:,.0f}")
print(f"RMSE:   Rp {rmse:,.0f}")
print(f"R²:     Rp {r2:,.4f}")
print()
print("=== Perbandingan ===")
print(f"Median harga rumah: Rp {y_test.median():,.0f}")
print(f"MAE sebagai persen median: {mae/y_test.median()*100:.1f}%")

=== Hasil Evaluasi Model Linear Regression ===
MAE:    Rp 199,130,531
RMSE:   Rp 377,716,712
R²:     Rp 0.7047

=== Perbandingan ===
Median harga rumah: Rp 200,000,000
MAE sebagai persen median: 99.6%


In [12]:
# Import Random Forest
from sklearn.ensemble import RandomForestRegressor

# Buat pipeline dengan Random Forest
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,   # 100 pohon keputusan
        random_state=42,    # biar hasil konsisten
        n_jobs=-1           # pakai semua core CPU (biar cepat)
    ))
])

# Training
print("Training Random Forest Regressor (mungkin 1-3 menit)...")
model_rf.fit(X_train, y_train)
print("Training selesai!")

Training Random Forest Regressor (mungkin 1-3 menit)...
Training selesai!


In [13]:
# Prediksi dengan Random Forest
y_pred_rf = model_rf.predict(X_test)

# Hitung metrik
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("=== Hasil Evaluasi Random Forest ===")
print(f"MAE:    Rp {mae_rf:,.0f}")
print(f"RMSE:   Rp {rmse_rf:,.0f}")
print(f"R²:     Rp {r2_rf:.4f}")

=== Hasil Evaluasi Random Forest ===
MAE:    Rp 78,658,134
RMSE:   Rp 198,613,190
R²:     Rp 0.9183


In [14]:
# Bandingkan kedua model
import pandas as pd

perbandingan = pd.DataFrame({
    'Model': ['Linear Regeression', 'Random Forest'],
    'MAE (Rp)': [mae, mae_rf],
    'RMSE (Rp)': [rmse, rmse_rf],
    'R²': [r2, r2_rf]
})

print("=== Perbandingan Model ===")
print(perbandingan.to_string(index=False))

# Tentukan pemenang
print()
if r2_rf > r2:
    print(f"PEMENANG: Random Forest (R² naik dari {r2:.4f} -> {r2_rf:.4f})")
else:
    print(f"PEMENANG: Linear Regression (R² {r2:.4f} > {r2_rf:.4f})")

=== Perbandingan Model ===
             Model     MAE (Rp)    RMSE (Rp)       R²
Linear Regeression 1.991305e+08 3.777167e+08 0.704692
     Random Forest 7.865813e+07 1.986132e+08 0.918349

PEMENANG: Random Forest (R² naik dari 0.7047 -> 0.9183)


In [15]:
# Import joblib untuk simpan model
import joblib

# Simpan model Random Forest ke folder models/
joblib.dump(model_rf, '../models/model_harga_rumah_rf.pkl')

print("Model disimpan ke: models/model_harga_rumah_rf.pkl")

Model disimpan ke: models/model_harga_rumah_rf.pkl


In [16]:
# Load model dari file
model_loaded = joblib.load('../models/model_harga_rumah_rf.pkl')

# Coba prediksi 5 rumah pertama di testing set
prediksi = model_loaded.predict(X_test.head(5))

print("=== Test Load Model ===")
print("Model berhasil di-load & prediksi jalan!")
print()
print("Contoh prediksi 5 rumah:")
for i in range(5):
    print(f"    Rumah {i+1}: Prediksi Rp {prediksi[i]:,.0f} | Aktual Rp {y_test.iloc[i]:,.0f}")

=== Test Load Model ===
Model berhasil di-load & prediksi jalan!

Contoh prediksi 5 rumah:
    Rumah 1: Prediksi Rp 411,894,567 | Aktual Rp 376,900,000
    Rumah 2: Prediksi Rp 148,894,179 | Aktual Rp 156,500,000
    Rumah 3: Prediksi Rp 1,447,822,705 | Aktual Rp 746,000,000
    Rumah 4: Prediksi Rp 184,405,000 | Aktual Rp 185,000,000
    Rumah 5: Prediksi Rp 423,953,663 | Aktual Rp 666,000,000


In [17]:
# Buat data rumah baru (fiktif)
rumah_baru = pd.DataFrame({
    'land_area_m2': [120, 200, 60],
    'building_area_m2': [90, 150, 45],
    'bedrooms': [3, 4, 2],
    'bathrooms': [2, 3, 1],
    'floors': [1, 2, 1],
    'total_units': [100, 50, 200],
    'latitude': [-6.2, -6.9, -7.2],
    'longitude': [106.8, 107.6, 112.7],
    'subsidy_status': ['Non-Subsidi', 'Non-Subsidi', 'Subsidi'],
    'province': ['DKI JAKARTA', 'JAWA BARAT', 'JAWA TIMUR'],
    'city_regency': ['KOTA JAKARTA SELATAN', 'KOTA BANDUNG', 'KOTA SURABAYA']
})

# Prediksi
prediksi_baru = model_loaded.predict(rumah_baru)

print("=== Prediksi Rumah Baru ===")
for i in range(len(rumah_baru)):
    print(f"\nRumah {i+1}:")
    print(f"  Luas: {rumah_baru.iloc[i]['land_area_m2']} m² tanah, {rumah_baru.iloc[i]['building_area_m2']} m² bangunan")
    print(f"  Kamar: {rumah_baru.iloc[i]['bedrooms']} tidur, {rumah_baru.iloc[i]['bathrooms']} mandi")
    print(f"  Lokasi: {rumah_baru.iloc[i]['province']}")
    print(f"  💰 PREDIKSI: Rp {prediksi_baru[i]:,.0f}")

=== Prediksi Rumah Baru ===

Rumah 1:
  Luas: 120 m² tanah, 90 m² bangunan
  Kamar: 3 tidur, 2 mandi
  Lokasi: DKI JAKARTA
  💰 PREDIKSI: Rp 681,784,982

Rumah 2:
  Luas: 200 m² tanah, 150 m² bangunan
  Kamar: 4 tidur, 3 mandi
  Lokasi: JAWA BARAT
  💰 PREDIKSI: Rp 2,731,911,778

Rumah 3:
  Luas: 60 m² tanah, 45 m² bangunan
  Kamar: 2 tidur, 1 mandi
  Lokasi: JAWA TIMUR
  💰 PREDIKSI: Rp 250,365,000
